# Test Result Count

Learning-Curve-Test fuer einen einzelnen Spieler. Dieses Notebook trainiert denselben Spieler mit mehreren Trainingsspiel-Anzahlen und evaluiert alle Modelle auf demselben Testsplit.

In [ ]:
import importlib
import sys
from pathlib import Path

for helper_dir in (Path.cwd(), Path.cwd() / "notebooks"):
    if (helper_dir / "test_result_utils.py").exists() and str(helper_dir) not in sys.path:
        sys.path.insert(0, str(helper_dir))

from IPython.display import Image, display

import test_result_utils

importlib.reload(test_result_utils)
from test_result_utils import ExperimentConfig, run_train_games_learning_curve


In [ ]:
COUNT_CONFIG = ExperimentConfig(
    player_usernames=("anatoltchirikov1951",),
    learning_curve_player_username="anatoltchirikov1951",
    learning_curve_train_games=(10, 25, 50, 75, 100, 150, 200, 300, 400, 500, 1000, 2000, 5000),
    learning_curve_save_models=False,
    max_games=25000,
    perf_type="classical",
    split_strategy="chronological",
    test_frac=0.01,
    val_frac_within_train=0.005,
    min_context_ply=10,
    learning_rate=2.5e-4,
    num_train_epochs=1,
    lora_rank=32,
    lora_alpha=64,
    target_modules="all-linear",
    require_train_games_per_player=True,
    results_root_name="results",
)

COUNT_RESULTS_DIR_TO_RESUME = ""

COUNT_CONFIG


In [ ]:
if COUNT_RESULTS_DIR_TO_RESUME:
    COUNT_RESULT = run_train_games_learning_curve(
        COUNT_CONFIG,
        resume_results_dir=COUNT_RESULTS_DIR_TO_RESUME,
    )
else:
    COUNT_RESULT = run_train_games_learning_curve(COUNT_CONFIG)

COUNT_SUMMARY = COUNT_RESULT["learning_curve_summary"]
COUNT_RESULT


In [ ]:
for row in COUNT_SUMMARY.get("curve_points", []):
    print(
        row["train_games_used"],
        "games | top1",
        round(row["finetuned_top1_accuracy"] * 100, 2),
        "% | delta",
        round(row["delta_top1_accuracy"] * 100, 2),
        "pp | mean rank",
        round(row["finetuned_mean_rank"], 3),
    )


In [ ]:
plot_path = COUNT_RESULT.get("plot_path") or COUNT_SUMMARY.get("plots", {}).get("train_games_learning_curve")
print(plot_path)
if plot_path and Path(plot_path).exists():
    display(Image(filename=plot_path))
